In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt

# from igraph import Graph
# from tqdm import tqdm

from src._const import conn_duck

query = """
INSTALL spatial;
LOAD spatial;

DROP TABLE IF EXISTS gla_oa;

CREATE TABLE IF NOT EXISTS oa_lookup AS SELECT * FROM 'data/data4report/lookup_census.csv';

CREATE TABLE IF NOT EXISTS gla_oa AS SELECT * FROM ST_Read('~/100_database/_boundary/_boundary_london/london_oa_2021/oa_london.shp');

SELECT gla_oa.OA21CD, oa_lookup.lsoa21cd, msoa21cd, ladcd
FROM gla_oa
LEFT JOIN oa_lookup
ON gla_oa.OA21CD = oa_lookup.oa21cd
"""

In [ ]:
df_gla_oa_lookup = conn_duck.query(query).df()

In [ ]:
gdf_gla_oa = gpd.read_file(
    "/Users/adamzh0u/100_database/_boundary/_boundary_london/london_lsoa_2021/lsoa_london_BFC.shp"
)

In [ ]:
import dask.dataframe as dd

df_trips = dd.read_parquet(
    "/Users/adamzh0u/100_database/_mobile/2021Nov_trj_oa.parquet"
)

In [ ]:
df_gla_points = df_trips[
    (df_trips["label"] != -1) & df_trips["o_oa"].isin(df_gla_oa_lookup["OA21CD"])
][["o_lat", "o_lon", "o_oa", "o_h9"]].compute()

In [ ]:
gdf_gla_oa.total_bounds

In [ ]:
gdf_gla = gdf_gla_oa.dissolve()

In [ ]:
import datashader as ds
import colorcet

cvs = ds.Canvas(
    plot_width=1500,
    plot_height=1200,
)
#  x_range=(df_gla_points.o_lon.min(), df_gla_points.o_lon.max()), y_range=(df_gla_points.o_lat.min(), df_gla_points.o_lat.max())
agg = cvs.points(df_gla_points, "o_lon", "o_lat")
img = ds.tf.shade(agg, cmap=colorcet.fire, how="log")
# img = img.rio.write_crs(27700)


# set background dark
plt.style.use("dark_background")

fig, ax = plt.subplots(figsize=(15, 12), dpi=400)
ax.axis("off")
ax.imshow(img.to_pil(), alpha=1)

fig.tight_layout()
# to pdf
fig.savefig(
    "fig/gla_oa_mobility_density_nov2021.pdf", bbox_inches="tight", pad_inches=0
)

In [ ]:
# Convert to Matplotlib figure

fig, ax = plt.subplots(figsize=(5, 3))
ax.imshow(
    img.to_pil(),
    extent=[
        df_gla_points.o_lon.min(),
        df_gla_points.o_lon.max(),
        df_gla_points.o_lat.min(),
        df_gla_points.o_lat.max(),
    ],
    alpha=0.6,
)
# cx.add_basemap(ax,crs="epsg:4326",source=cx.providers.CartoDB.Positron,zorder=-1)
# gdf_gla.to_crs(epsg=4326).boundary.plot(ax=ax, edgecolor='black', linewidth=0.1, alpha=0.5, zorder=-1)